# Lab 2: Feature Extraction (MSCOCO with 5 Captions per Image)

In [6]:

import os
import numpy as np
from tqdm import tqdm
from pycocotools.coco import COCO
from PIL import Image
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from transformers import BertTokenizer, BertModel

# Paths (adjust as needed)
data_dir = "/home/BTECH_7TH_SEM/MS-COCO"
annFile = os.path.join(data_dir, "annotations_trainval2017/annotations/captions_val2017.json")
annFile_labels = os.path.join(data_dir, "annotations_trainval2017/annotations/instances_val2017.json")
img_dir = os.path.join(data_dir, "val2017")

# Load COCO APIs
coco_caps = COCO(annFile)
coco_labels = COCO(annFile_labels)
img_ids = coco_caps.getImgIds()[:5000]  # subset of 5000 images

# Image feature extractor (ResNet50)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet = models.resnet50(pretrained=True)
resnet = torch.nn.Sequential(*list(resnet.children())[:-1])  # remove final layer
resnet.eval().to(device)

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

# BERT feature extractor
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased").to(device)
bert.eval()

def extract_image_feature(img_path):
    image = Image.open(img_path).convert("RGB")
    img_t = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = resnet(img_t).squeeze().cpu().numpy()
    return feat

def extract_caption_features(captions):
    feats = []
    for c in captions:
        inputs = tokenizer(c, return_tensors="pt", truncation=True, padding=True, max_length=30).to(device)
        with torch.no_grad():
            outputs = bert(**inputs)
            cls_emb = outputs.last_hidden_state[:,0,:].squeeze().cpu().numpy()
        feats.append(cls_emb)
    return np.mean(feats, axis=0)  # average over 5 captions

# Arrays
image_features, caption_features, labels = [], [], []

for img_id in tqdm(img_ids):
    try:
        img_info = coco_caps.loadImgs(img_id)[0]
        img_path = os.path.join(img_dir, img_info["file_name"])
        
        # Skip if image feature extraction fails
        img_feat = extract_image_feature(img_path)
        if img_feat is None:
            continue

        # Check if we have captions
        ann_ids = coco_caps.getAnnIds(imgIds=img_id)
        if not ann_ids:
            continue
            
        anns = coco_caps.loadAnns(ann_ids)
        caps = [a["caption"] for a in anns[:5]]
        if len(caps) < 5:  # Skip if less than 5 captions
            continue
            
        cap_feat = extract_caption_features(caps)

        # Check for labels
        ann_ids_labels = coco_labels.getAnnIds(imgIds=img_id)
        if not ann_ids_labels:
            continue
            
        anns_labels = coco_labels.loadAnns(ann_ids_labels)
        cats = [a["category_id"] for a in anns_labels]
        
        multi_hot = np.zeros(80)
        for c in cats:
            if 0 < c <= 80:  # Ensure valid category ID
                multi_hot[c-1] = 1

        image_features.append(img_feat)
        caption_features.append(cap_feat)
        labels.append(multi_hot)
        
    except Exception as e:
        print(f"Error processing image {img_id}: {str(e)}")
        continue

image_features = np.array(image_features)
caption_features = np.array(caption_features)
labels = np.array(labels)

print("Image features:", image_features.shape)
print("Caption features:", caption_features.shape)
print("Labels:", labels.shape)

# Save
os.makedirs("coco_features", exist_ok=True)
np.save("coco_features/image_features.npy", image_features)
np.save("coco_features/caption_features.npy", caption_features)
np.save("coco_features/labels.npy", labels)


loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
loading annotations into memory...
Done (t=0.17s)
creating index...
index created!


100%|██████████| 5000/5000 [01:17<00:00, 64.82it/s]


Image features: (4952, 2048)
Caption features: (4952, 768)
Labels: (4952, 80)
